In [8]:
TEXT = """low low low low low lower lower widest widest widest newest newest newest newest newest newest"""
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
import regex as re
text_iter = re.finditer(PAT, TEXT)
text_iter = re.split(" ", TEXT)
pretoken_dict = {}
for t in text_iter:
    # encoded = tuple(t.encode("utf-8"))
    encoded = tuple(bytes([b]) for b in t.encode("utf-8"))
    if encoded not in pretoken_dict:
        pretoken_dict[encoded] = 1
    else:
        pretoken_dict[encoded] += 1

In [9]:
pretoken_dict

{(b'l', b'o', b'w'): 5,
 (b'l', b'o', b'w', b'e', b'r'): 2,
 (b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 6}

In [10]:
ADDED_VOCAB = 6
new_vocab_list = []
for i in range(ADDED_VOCAB):
    paired_dict = {}
    for k, v in pretoken_dict.items():
        for i in range(len(k)-1):
            pair = k[i], k[i+1]
            if pair not in paired_dict:
                paired_dict[pair] = v
            else:
                paired_dict[pair] += v

    max_value = max(paired_dict.values())
    paired_bytes = max([k for k, v in paired_dict.items() if v == max_value])

    new_vocab = b"".join(paired_bytes)
    new_vocab_list.append(new_vocab)

    for k in list(pretoken_dict.keys()):
        if paired_bytes in zip(k, k[1:]):
            v = pretoken_dict.pop(k)
            new_k = []
            i = 0
            while i < len(k):
                if i < len(k) - 1 and (k[i], k[i + 1]) == paired_bytes:
                    new_k.append(new_vocab)
                    i += 2
                else:
                    new_k.append(k[i])
                    i += 1
            pretoken_dict[tuple(new_k)] = v

In [11]:
pretoken_dict

{(b'w', b'i', b'd', b'est'): 3,
 (b'low',): 5,
 (b'low', b'e', b'r'): 2,
 (b'ne', b'west'): 6}

In [12]:
new_vocab_list

[b'st', b'est', b'ow', b'low', b'west', b'ne']